In [1]:
import pandas as pd
import numpy as np
import pandas as pd
import numpy as np
import sklearn

In [2]:
df=pd.read_csv("/content/drive/MyDrive/UoB_Interview_Task/dummy_data/cleaned_data.csv")

In [3]:
df.head()

,subject_id,hadm_id,admission_type,admission_location,insurance,language,marital_status,race,hospital_expire_flag,ccs_seq,...,hour_cos,dow_sin,dow_cos,mon_sin,mon_cos,admission_index,days_since_prev_admission,days_since_first_admission,num_prior_admissions,has_prior_admission
0,10000690,26504700,EW EMER.,EMERGENCY ROOM,Medicare,English,WIDOWED,WHITE,0,"['CIR019', 'CIR019', 'CIR019', 'CIR003', 'CIR0...",...,8.660254e-01,-0.433884,-0.900969,1.224647e-16,-1.000000e+00,0,0.000000,0.000000,0,0
1,10000690,23280645,EW EMER.,EMERGENCY ROOM,Medicare,English,WIDOWED,WHITE,0,"['CIR019', 'CIR019', 'RSP002', 'CIR017', 'END0...",...,2.588190e-01,0.974928,-0.222521,-8.660254e-01,-5.000000e-01,1,71.170833,75.709722,1,1
2,10000690,25860671,EW EMER.,EMERGENCY ROOM,Medicare,English,WIDOWED,WHITE,0,"['RSP010', 'CIR019', 'RSP012', 'RSP012', 'GEN0...",...,-1.836970e-16,0.000000,1.000000,-8.660254e-01,5.000000e-01,2,39.175000,122.636111,2,1
3,10000690,26146595,EW EMER.,EMERGENCY ROOM,Medicare,English,WIDOWED,WHITE,0,"['DIG012', 'DIG012', 'DIG012', 'GEN002', 'CIR0...",...,9.659258e-01,-0.433884,-0.900969,0.000000e+00,1.000000e+00,3,442.413194,574.870833,3,1
4,10001919,29897682,SURGICAL SAME DAY ADMISSION,PHYSICIAN REFERRAL,Private,English,MARRIED,OTHER,0,"['NEO013', 'NEO070', 'CIR033', 'DIG024', 'XXX0...",...,1.000000e+00,0.433884,-0.900969,1.000000e+00,6.123234e-17,0,0.000000,0.000000,0,0


In [4]:
df["ed_duration"] = df["ed_duration"].fillna(0)

In [5]:
df = df.fillna("missing")

In [6]:
## could include cumulative length of prior admissions later on

def to_text(row):
    # Build a brief, stable prompt. Keep order + content consistent.
    ccs = row["ccs_seq"]
    if isinstance(ccs, (list, tuple)):
        ccs_str = ", ".join(ccs[:10])
    else:
        ccs_str = ", ".join(str(ccs).split(",")[:10])

    return (
        f"Age: {row['age_at_admission']}. Sex: {row['gender']}. Race: {row['race']}."
        f"Admission Type: {row['admission_type']}. Admission location:{row['admission_location']}."
        f"Insurance: {row['insurance']}. Language: {row['language']}. Marital status: {row['marital_status']}."
        f"Diagnosis Sequence: {ccs_str}."
        f"Prior admissions : {row['num_prior_admissions']}. Days since first admission: {row['days_since_first_admission']}."
        f"Days since previous admission: {row['days_since_prev_admission']}."
    )

In [7]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

df["text"] = df.apply(to_text, axis=1)
df["label"] = df["hospital_expire_flag"].astype(int)

# Group split by subject_id
gss = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["subject_id"]))
df_train = df.iloc[train_idx].reset_index(drop=True)
df_test  = df.iloc[test_idx].reset_index(drop=True)

# Make a val split from train
gss2 = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=43)
tr_idx, val_idx = next(gss2.split(df_train, groups=df_train["subject_id"]))
df_tr = df_train.iloc[tr_idx].reset_index(drop=True)
df_val = df_train.iloc[val_idx].reset_index(drop=True)

In [8]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"  # good default for MIMIC-style text

tok = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tokenize(batch):
    return tok(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=512,  # keep short; increase to 512 if you have VRAM
    )

ds_tr  = Dataset.from_pandas(df_tr[["text", "label", "race"]])
ds_val = Dataset.from_pandas(df_val[["text", "label", "race"]])
ds_te  = Dataset.from_pandas(df_test[["text", "label", "race"]])

ds_tr  = ds_tr.map(tokenize, batched=True)
ds_val = ds_val.map(tokenize, batched=True)
ds_te  = ds_te.map(tokenize, batched=True)

cols = ["input_ids", "attention_mask", "label"]
ds_tr.set_format(type="torch", columns=cols)
ds_val.set_format(type="torch", columns=cols)
ds_te.set_format(type="torch", columns=cols)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/7590 [00:00<?, ? examples/s]

Map:   0%|          | 0/1949 [00:00<?, ? examples/s]

Map:   0%|          | 0/2289 [00:00<?, ? examples/s]

In [9]:
!pip install evaluate

In [10]:
import torch, evaluate
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

num_labels = 2

base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32
)

# (Optional) 4-bit QLoRA:
USE_QLORA = False  # set True if using bitsandbytes + CUDA
if USE_QLORA:
    from transformers import BitsAndBytesConfig
    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True,
                                    bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16)
    base_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=num_labels,
        quantization_config=bnb_config,
        device_map="auto"
    )
    base_model = prepare_model_for_kbit_training(base_model)

# LoRA config (tune query/key/value projections; keep it light)
lora_cfg = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS",  # sequence classification
    target_modules=["query", "key", "value", "dense"]  # safe defaults; model-dependent
)
model = get_peft_model(base_model, lora_cfg)
model.print_trainable_parameters()


`torch_dtype` is deprecated! Use `dtype` instead!
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at emilyalsentzer/Bio_ClinicalBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 1,340,930 || all params: 109,652,740 || trainable%: 1.2229


In [11]:
import numpy as np
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support, accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    preds = (probs >= 0.5).astype(int)

    auc = roc_auc_score(labels, probs)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
    acc = accuracy_score(labels, preds)
    return {"auc_roc": auc, "precision": prec, "recall": rec, "f1": f1, "accuracy": acc}


In [12]:
# Compute weights from the training set
pos_rate = df_tr["label"].mean()
w_pos = 0.5 / max(pos_rate, 1e-6)
w_neg = 0.5 / max(1 - pos_rate, 1e-6)
class_weights = torch.tensor([w_neg, w_pos]).to("cuda" if torch.cuda.is_available() else "cpu")

def weighted_loss_func(model, inputs, return_outputs=False):
    labels = inputs.get("labels")
    outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})
    loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights)
    loss = loss_fct(outputs.logits.view(-1, 2), labels.view(-1))
    return (loss, outputs) if return_outputs else loss


In [17]:
!pip install -U transformers accelerate datasets evaluate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 42.1 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.56.0
    Uninstalling transformers-4.56.0:
      Successfully uninstalled transformers-4.56.0


In [16]:
training_args = TrainingArguments(
    output_dir="./mortality_bioclincbert_lora",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,          # a bit higher for LoRA adapters
    num_train_epochs=6,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="auc_roc",
    greater_is_better=True,
    logging_steps=50,
    bf16=torch.cuda.is_available(),  # if your GPU supports bf16
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_tr,
    eval_dataset=ds_val,
    tokenizer=tok,
    compute_metrics=compute_metrics,
    # loss_func=weighted_loss_func  # If using class weights with HF ≥ 4.43; otherwise override Trainer subclass
)

trainer.train()
val_metrics = trainer.evaluate()
print(val_metrics)


/tmp/ipython-input-4254005119.py:21: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: acd446 (acd446-university-of-birmingham) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Auc Roc,Precision,Recall,F1,Accuracy
1,0.108300,0.104536,0.725579,0.000000,0.000000,0.000000,0.978450


Epoch,Training Loss,Validation Loss,Auc Roc,Precision,Recall,F1,Accuracy
1,0.108300,0.104536,0.725579,0.000000,0.000000,0.000000,0.978450
2,0.084400,0.120311,0.745487,0.000000,0.000000,0.000000,0.978450
3,0.107100,0.076823,0.909925,0.000000,0.000000,0.000000,0.978450
4,0.096600,0.071514,0.930244,0.000000,0.000000,0.000000,0.977937
5,0.100000,0.072279,0.925993,0.250000,0.047619,0.080000,0.976398
6,0.067700,0.071908,0.926293,0.000000,0.000000,0.000000,0.976911


{'eval_loss': 0.07151419669389725, 'eval_auc_roc': 0.9302444627562613, 'eval_precision': 0.0, 'eval_recall': 0.0, 'eval_f1': 0.0, 'eval_accuracy': 0.9779374037968189, 'eval_runtime': 99.8052, 'eval_samples_per_second': 19.528, 'eval_steps_per_second': 0.611, 'epoch': 6.0}


In [20]:
# Overall test metrics
test_metrics = trainer.evaluate(ds_te)
print("Test:", test_metrics)

# Subgroup eval by race
import pandas as pd
from torch.utils.data import DataLoader

def predict_probs(dataset):
    preds = trainer.predict(dataset)
    probs = torch.softmax(torch.tensor(preds.predictions), dim=-1)[:, 1].numpy()
    return probs, preds.label_ids

probs, labels = predict_probs(ds_te)
df_out = pd.DataFrame({
    "race": df_test["race"].values,
    "y_true": labels,
    "y_prob": probs,
    "y_pred": (probs >= 0.2).astype(int)
})

from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
rows = []
for grp, g in df_out.groupby("race"):
    if g["y_true"].nunique() == 2:  # AUC needs both classes
        auc = roc_auc_score(g["y_true"], g["y_prob"])
    else:
        auc = np.nan
    rows.append({
        "race": grp,
        "n": len(g),
        "auc_roc": auc,
        "f1": f1_score(g["y_true"], g["y_pred"], zero_division=0),
        "precision": precision_score(g["y_true"], g["y_pred"], zero_division=0),
        "recall": recall_score(g["y_true"], g["y_pred"], zero_division=0),
    })
pd.DataFrame(rows).sort_values("n", ascending=False)


Test: {'eval_loss': 0.07681765407323837, 'eval_auc_roc': 0.8982067329184004, 'eval_precision': 0.0, 'eval_recall': 0.0, 'eval_f1': 0.0, 'eval_accuracy': 0.9772826561817387, 'eval_runtime': 117.8946, 'eval_samples_per_second': 19.416, 'eval_steps_per_second': 0.611, 'epoch': 6.0}


,race,n,auc_roc,f1,precision,recall
26,WHITE,1431,0.892952,0.373626,0.320755,0.447368
7,BLACK/AFRICAN AMERICAN,324,0.996894,0.444444,0.285714,1.000000
20,OTHER,109,1.000000,0.666667,0.500000,1.000000
25,UNKNOWN,58,0.949074,0.250000,0.250000,0.250000
3,ASIAN - CHINESE,58,0.991228,0.400000,0.250000,1.000000
28,WHITE - OTHER EUROPEAN,51,NaN,0.000000,0.000000,0.000000
16,HISPANIC/LATINO - PUERTO RICAN,34,0.106061,0.000000,0.000000,0.000000
29,WHITE - RUSSIAN,31,0.933333,0.000000,0.000000,0.000000
8,BLACK/CAPE VERDEAN,30,0.551724,0.000000,0.000000,0.000000
1,ASIAN,30,NaN,0.000000,0.000000,0.000000


In [23]:
df_out.to_csv("/content/drive/MyDrive/UoB_Interview_Task/output/llm.csv")

In [18]:
# Save
adapter_path = "./mortality_bioclincbert_lora/best_adapter"
trainer.model.save_pretrained(adapter_path)
tok.save_pretrained(adapter_path)

# Load later (in another notebook)
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import PeftModel

base = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
tok  = AutoTokenizer.from_pretrained(adapter_path)
model = PeftModel.from_pretrained(base, adapter_path)
model.eval()


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at emilyalsentzer/Bio_ClinicalBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): BertForSequenceClassification(
      (bert): BertModel(
        (embeddings): BertEmbeddings(
          (word_embeddings): Embedding(28996, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (token_type_embeddings): Embedding(2, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (encoder): BertEncoder(
          (layer): ModuleList(
            (0-11): 12 x BertLayer(
              (attention): BertAttention(
                (self): BertSdpaSelfAttention(
                  (query): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.05, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (defaul